# Stage 10 — Base ASAG Integrated Pipeline

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 10 — Base ASAG Pipeline Integration (`BaseASAGPipeline`)  

---

### Research Foundation & Methodological Context
This notebook integrates and demonstrates the complete classical **Base ASAG Pipeline (System 2)**, inspired by:
> **Basak, R., Naskar, S. K., & Gelbukh, A. (2019).**  
> *"Short-answer grading using textual entailment."*  
> Journal of Intelligent & Fuzzy Systems, 36(5), 4909–4919. DOI: 10.3233/JIFS-179038.

The Base ASAG system evaluates student answers through an interpretable, transparent pipeline of classical NLP stages:
$$\text{Raw Text} \xrightarrow{\text{Stage 3}} \text{Preprocessing} \xrightarrow{\text{Stage 4}} \text{Parsing} \xrightarrow{\text{Stage 5}} \text{Triples} \xrightarrow{\text{Stage 6}} \text{Semantics} \xrightarrow{\text{Stage 7}} \text{Matching} \xrightarrow{\text{Stage 8}} \text{Entailment} \xrightarrow{\text{Stage 9}} \text{Scoring}$$

### Research Boundaries & Hypothesis
- **System 2 (Base ASAG):** Evaluates reference proposition coverage. Extra student information is preserved in `unmatched_student_relations` with **strictly zero score modification** ($0.0$ bonus, $0.0$ penalty).
- **Research Question:** *"Can an interpretable proposition-level extension of a classical semantic/dependency-based ASAG system evaluate unmatched student information as correct additional, irrelevant, or incorrect information, and does incorporating this evaluation improve agreement with human grading?"*
- **Research Hypothesis:** The proposed extension is an experimental hypothesis to be evaluated in Stages 11–16. We do **not** assume it improves grading.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> Synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We initialize `BaseASAGPipeline` (System 2) and `TfidfCosineBaseline` (System 1) to compare lexical bag-of-words against relational semantic/dependency grading.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.pipeline import BaseASAGPipeline, BaseASAGPipelineResult
from baseline.tfidf_cosine.baseline import TfidfCosineBaseline

# Initialize Base ASAG Pipeline (System 2)
base_pipeline = BaseASAGPipeline()

# Initialize Baseline (System 1)
tfidf_baseline = TfidfCosineBaseline()

print("Base ASAG Pipeline initialized successfully with all Stages (3–9) connected.")
print(f"  Matcher alignment threshold: {base_pipeline.matcher.alignment_threshold}")
print(f"  Entailment threshold: {base_pipeline.analyzer.entailment_threshold}")
print(f"  Scorer weights: ENTAILED=1.0, UNCERTAIN={base_pipeline.scorer.uncertain_weight}, CONTRADICTED=0.0, MISSING=0.0")

Base ASAG Pipeline initialized successfully with all Stages (3–9) connected.
  Matcher alignment threshold: 0.5
  Entailment threshold: 0.7
  Scorer weights: ENTAILED=1.0, UNCERTAIN=0.5, CONTRADICTED=0.0, MISSING=0.0


## 2. Demonstration Dataset

We use the standard motivating demonstration example from `GEMINI.md`:
- **Question:** *"What is the purpose of a primary key?"*
- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*

We define 7 synthetic student responses representing distinct evaluation conditions:
1. **Identical Answer:** Exact match to the reference answer.
2. **Student A (Paraphrased):** Uses *"every row"* instead of *"each record"*.
3. **Student B (Correct Additional):** Fully correct answer + correct extra property (*"cannot contain NULL values"*).
4. **Student C (Incorrect Additional):** Fully correct answer + misconception (*"can contain duplicate values"*).
5. **Student D (Irrelevant Additional):** Fully correct answer + off-topic fact (*"SQL was developed in the 1970s"*).
6. **Contradicted Answer:** Explicit negation (*"never identifies each record"*).
7. **Unrelated Answer:** Off-topic statement (*"Photosynthesis requires sunlight and water."*).

In [2]:
question = "What is the purpose of a primary key?"
reference_answer = "A primary key uniquely identifies each record in a table."

student_cases = [
    {
        "label": "Identical",
        "text": "A primary key uniquely identifies each record in a table.",
        "human_expectation": "Full credit (1.00)",
        "category": "Fully Correct",
    },
    {
        "label": "Student A (Paraphrased)",
        "text": "A primary key uniquely identifies every row in a table.",
        "human_expectation": "Full credit (1.00) under ideal semantics",
        "category": "Paraphrase",
    },
    {
        "label": "Student B (Correct Extra)",
        "text": "A primary key uniquely identifies each record in a table and cannot contain NULL values.",
        "human_expectation": "Full credit (1.00) or minor bonus",
        "category": "Correct Additional",
    },
    {
        "label": "Student C (Incorrect Extra)",
        "text": "A primary key uniquely identifies each record in a table and can contain duplicate values.",
        "human_expectation": "Penalty / deduction due to misconception",
        "category": "Incorrect Additional",
    },
    {
        "label": "Student D (Irrelevant Extra)",
        "text": "A primary key uniquely identifies each record in a table. SQL was developed in the 1970s.",
        "human_expectation": "Full credit (1.00) or minor deduction for irrelevance",
        "category": "Irrelevant Additional",
    },
    {
        "label": "Contradicted",
        "text": "A primary key never identifies each record in a table.",
        "human_expectation": "Zero credit (0.00)",
        "category": "Direct Contradiction",
    },
    {
        "label": "Unrelated",
        "text": "Photosynthesis requires sunlight and water.",
        "human_expectation": "Zero credit (0.00)",
        "category": "Completely Unrelated",
    },
]

## 3. End-to-End Pipeline Execution

We grade each student response with a single call to `base_pipeline.grade()`.
Each call executes Stages 3 to 9 in deterministic sequence and returns a structured `BaseASAGPipelineResult`.

In [3]:
results = []

for case in student_cases:
    # Run System 2: Base ASAG Pipeline
    pipe_res = base_pipeline.grade(question, reference_answer, case["text"])
    
    # Run System 1: TF-IDF Baseline for comparison
    tfidf_score = tfidf_baseline.similarity(reference_answer, case["text"])
    
    results.append({
        "label": case["label"],
        "text": case["text"],
        "category": case["category"],
        "human_expectation": case["human_expectation"],
        "tfidf_score": round(tfidf_score, 4),
        "pipe_result": pipe_res,
        "base_score": pipe_res.normalized_score,
        "scaled_score_5": pipe_res.scale_score(5.0),
        "entailed": pipe_res.entailed_count,
        "uncertain": pipe_res.uncertain_count,
        "contradicted": pipe_res.contradicted_count,
        "missing": pipe_res.missing_count,
        "unmatched_student": pipe_res.unmatched_student_count,
    })

print(f"Successfully graded all {len(results)} demonstration answers through BaseASAGPipeline.")

Successfully graded all 7 demonstration answers through BaseASAGPipeline.


## 4. Comparative Evaluation Table (System 1 vs System 2)

Below is the comparative grading table summarizing the behavior of both systems across the demonstration scenarios.

In [4]:
df_summary = pd.DataFrame([
    {
        "Student Case": r["label"],
        "TF-IDF Baseline (Sys 1)": f"{r['tfidf_score']:.4f}",
        "Base ASAG [0-1] (Sys 2)": f"{r['base_score']:.4f}",
        "Base ASAG (5.0 scale)": f"{r['scaled_score_5']:.2f}",
        "Entailed": r["entailed"],
        "Uncertain": r["uncertain"],
        "Contradicted": r["contradicted"],
        "Missing": r["missing"],
        "Unmatched Student Rels": r["unmatched_student"],
        "Human Expectation": r["human_expectation"],
    }
    for r in results
])

pd.set_option("display.max_colwidth", None)
display(df_summary)

,Student Case,TF-IDF Baseline (Sys 1),Base ASAG [0-1] (Sys 2),Base ASAG (5.0 scale),Entailed,Uncertain,Contradicted,Missing,Unmatched Student Rels,Human Expectation
0,Identical,1.0000,1.0000,5.00,1,0,0,0,0,Full credit (1.00)
1,Student A (Paraphrased),0.7168,0.5000,2.50,0,1,0,0,0,Full credit (1.00) under ideal semantics
2,Student B (Correct Extra),0.7406,1.0000,5.00,1,0,0,0,1,Full credit (1.00) or minor bonus
3,Student C (Incorrect Extra),0.7406,1.0000,5.00,1,0,0,0,1,Penalty / deduction due to misconception
4,Student D (Irrelevant Extra),0.7524,1.0000,5.00,1,0,0,0,1,Full credit (1.00) or minor deduction for irrelevance
5,Contradicted,0.8478,0.0000,0.00,0,0,1,0,0,Zero credit (0.00)
6,Unrelated,0.0000,0.0000,0.00,0,0,0,1,1,Zero credit (0.00)


## 5. Traceable Inspection of Intermediate Pipeline Representations

Because `BaseASAGPipelineResult` preserves all intermediate data structures, we can inspect every decision step from parsing to scoring.

In [5]:
# Student B: Correct Answer + Correct Extra Information
res_b = results[2]["pipe_result"]

print("=== STUDENT B DETAILED DIAGNOSTICS ===")
print(f"Student Text: {res_b.student_answer}")
print(f"Normalized Score: {res_b.normalized_score} / 1.0")
print(f"Reference Relations ({len(res_b.reference_triples.canonical_relations)}):", [r.to_tuple() for r in res_b.reference_triples.canonical_relations])
print(f"Student Relations ({len(res_b.student_triples.canonical_relations)}):", [r.to_tuple() for r in res_b.student_triples.canonical_relations])

print("\nMatched Relations:")
for m in res_b.matched_relations:
    print(f"  Ref: {m.reference_relation.to_tuple()} <--> Stu: {m.student_relation.to_tuple()} (Triple Score: {m.triple_score:.3f})")

print("\nEntailment Evaluation:")
for e in res_b.entailment_results:
    print(f"  Decision: {e.relation} (Evidence Score: {e.confidence:.3f}) | Student Polarity: {e.student_polarity}")

print("\nUnmatched Student Relations (Preserved for Stage 11–14):")
for u in res_b.unmatched_student_relations:
    print(f"  Extra Student Proposition: {u.to_tuple()} [Type: {u.relation_type}]")
    print("  Base ASAG Score Effect: Exactly 0.00 (Neutral)")

=== STUDENT B DETAILED DIAGNOSTICS ===
Student Text: A primary key uniquely identifies each record in a table and cannot contain NULL values.
Normalized Score: 1.0 / 1.0
Reference Relations (1): [('key', 'identifies', 'record')]
Student Relations (2): [('key', 'identifies', 'record'), ('key', 'contain', 'values')]

Matched Relations:
  Ref: ('key', 'identifies', 'record') <--> Stu: ('key', 'identifies', 'record') (Triple Score: 1.000)

Entailment Evaluation:
  Decision: ENTAILS (Evidence Score: 1.000) | Student Polarity: POSITIVE

Unmatched Student Relations (Preserved for Stage 11–14):
  Extra Student Proposition: ('key', 'contain', 'values') [Type: SPO]
  Base ASAG Score Effect: Exactly 0.00 (Neutral)


In [6]:
# Student C: Correct Answer + Incorrect Extra Information (Misconception)
res_c = results[3]["pipe_result"]

print("=== STUDENT C DETAILED DIAGNOSTICS ===")
print(f"Student Text: {res_c.student_answer}")
print(f"Normalized Score: {res_c.normalized_score} / 1.0 (Identical to Student B!)")

print("\nUnmatched Student Relations (Misconception):")
for u in res_c.unmatched_student_relations:
    print(f"  Extra Student Proposition: {u.to_tuple()} [Type: {u.relation_type}]")
    print("  Base ASAG Score Effect: 0.00 (Unpenalized)")

print("\nKEY OBSERVATION:")
print("Base ASAG awards Student C 1.00 because the reference proposition coverage is complete.")
print("The misconception ('can contain duplicate values') is ignored by Base ASAG scoring.")

=== STUDENT C DETAILED DIAGNOSTICS ===
Student Text: A primary key uniquely identifies each record in a table and can contain duplicate values.
Normalized Score: 1.0 / 1.0 (Identical to Student B!)

Unmatched Student Relations (Misconception):
  Extra Student Proposition: ('key', 'contain', 'values') [Type: SPO]
  Base ASAG Score Effect: 0.00 (Unpenalized)

KEY OBSERVATION:
Base ASAG awards Student C 1.00 because the reference proposition coverage is complete.
The misconception ('can contain duplicate values') is ignored by Base ASAG scoring.


In [7]:
# Student D: Correct Answer + Irrelevant Extra Information
res_d = results[4]["pipe_result"]

print("=== STUDENT D DETAILED DIAGNOSTICS ===")
print(f"Student Text: {res_d.student_answer}")
print(f"Normalized Score: {res_d.normalized_score} / 1.0 (Identical to Students B and C)")

print("\nUnmatched Student Relations (Irrelevant Fact):")
for u in res_d.unmatched_student_relations:
    print(f"  Extra Student Proposition: {u.to_tuple()} [Type: {u.relation_type}]")
    print("  Base ASAG Score Effect: 0.00 (Neutral)")

print("\nKEY OBSERVATION:")
print("Base ASAG cannot distinguish whether unmatched information is relevant, correct, or erroneous.")

=== STUDENT D DETAILED DIAGNOSTICS ===
Student Text: A primary key uniquely identifies each record in a table. SQL was developed in the 1970s.
Normalized Score: 1.0 / 1.0 (Identical to Students B and C)

Unmatched Student Relations (Irrelevant Fact):
  Extra Student Proposition: ('SQL', 'developed', None) [Type: PASSIVE_SP]
  Base ASAG Score Effect: 0.00 (Neutral)

KEY OBSERVATION:
Base ASAG cannot distinguish whether unmatched information is relevant, correct, or erroneous.


## 6. Lexical Resource Observation: Paraphrased Answer (Student A)

Student A wrote: *"A primary key uniquely identifies every row in a table."*  
In relational databases, *"record"* and *"row"* are synonymous in common usage.  
However, in standard WordNet:
- `record.n.01` (document / evidence) vs `row.n.01` (linear arrangement)
- WordNet Path Similarity: `0.125`

Because $0.125 < 0.50$ (minimum slot similarity), Stage 8 classifies this proposition as **`UNCERTAIN`**, awarding partial credit ($0.50$). This is an authentic limitation of classical lexical taxonomy matching, preserved for honest analysis rather than obscured with ad-hoc heuristics.

In [8]:
res_a = results[1]["pipe_result"]

print("=== STUDENT A (PARAPHRASE) DETAILED DIAGNOSTICS ===")
print(f"Student Text: {res_a.student_answer}")
print(f"Normalized Score: {res_a.normalized_score} / 1.0")
print("Entailment Record:")
for e in res_a.entailment_results:
    print(f"  Decision: {e.relation} | Evidence Score: {e.confidence:.4f}")
    print(f"  Diagnostic Reasoning: {e.reasons}")

=== STUDENT A (PARAPHRASE) DETAILED DIAGNOSTICS ===
Student Text: A primary key uniquely identifies every row in a table.
Normalized Score: 0.5 / 1.0
Entailment Record:
  Decision: UNCERTAIN | Evidence Score: 0.7083
  Diagnostic Reasoning: ['Polarity is compatible (both propositions are POSITIVE), but semantic support is inconclusive.', "Object similarity (0.1250) for 'record' <-> 'row' is below slot threshold (0.50)."]


## 7. Research Insights & Motivation for Proposed Extension (Stage 11)

### What Base ASAG Accomplishes:
1. **Structural Interpretability:** Scores are computed from traceable grammatical relationships, not opaque black-box weights.
2. **Polarity Sensitivity:** Explicit negation (*"never identifies"*) correctly prevents false credit.
3. **Role Preservation:** Subjects, verbs, and objects must correspond grammatically.

### The Critical Limitation of Base ASAG:
Observe that **Students B, C, and D all receive an identical Base ASAG score of 1.00**:
$$\text{Score}(\text{Student B}) = \text{Score}(\text{Student C}) = \text{Score}(\text{Student D}) = 1.00$$

- **Student B** added a **correct additional fact** (*cannot contain NULL values*).
- **Student C** added an **incorrect misconception** (*can contain duplicate values*).
- **Student D** added an **irrelevant fact** (*SQL was developed in the 1970s*).

Because classical entailment ASAG (Basak et al., 2019) focuses strictly on whether the reference answer is entailed by the student answer, **unmatched student information is ignored during scoring**.

### Foundation for Stage 11 (Proposed ASAG):
This exact limitation directly motivates the proposed extension:
> **Research Question:**  
> *"Can an interpretable proposition-level extension of a classical semantic/dependency-based ASAG system evaluate unmatched student information as correct additional, irrelevant, or incorrect information, and does incorporating this evaluation improve agreement with human grading?"*

In **Stage 11**, we will formalize the extraction and partitioning of:
1. **Matched Information:** Student propositions successfully aligned with reference propositions.
2. **Unmatched Student Information:** Student propositions outside the reference scope, preparing them for proposition-level evaluation in Stages 12–14.